In [5]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
df = pd.read_excel('../data/dataset_assurance_ML.xlsx')
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig')
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig')
print(df.shape)

(500, 27)


In [14]:
TARGET = 'Résiliation'
print(f"{df[TARGET].value_counts().iloc[0]} clients restent (0) et {df[TARGET].value_counts().iloc[1]} clients résilient (1) --> {(df[TARGET].value_counts().iloc[1] / df[TARGET].value_counts().sum())*100} %")

450 clients restent (0) et 50 clients résilient (1) --> 10.0 %


In [15]:
crosstab_statut_resiliation = pd.crosstab(df['Statut Contrat'], df[TARGET])
print(crosstab_statut_resiliation)

print(
    "\nLe statut du contrat révèle la cible à 100 % : chaque statut correspond à une seule valeur de Résiliation. "
    "Il ne faut donc pas utiliser 'Statut Contrat' pour entraîner le modèle, car ce serait une fuite de cible : "
    "au moment de prédire, le statut associé à la résiliation n'est pas encore connu."
)

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14

Le statut du contrat révèle la cible à 100 % : chaque statut correspond à une seule valeur de Résiliation. Il ne faut donc pas utiliser 'Statut Contrat' pour entraîner le modèle, car ce serait une fuite de cible : au moment de prédire, le statut associé à la résiliation n'est pas encore connu.


In [16]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)',
'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)',
'Montant Sinistres (€)', 'Score Risque (0-100)']
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']
X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

(500, 12) (500,)


In [17]:
correlations = X[num_cols].corrwith(y).round(3).sort_values(ascending=False)
print(correlations)

taux_resiliation_sinistre = (
    df.groupby('Dernier Sinistre')[TARGET]
    .mean()
    .round(2)
    .sort_values()
)
print(taux_resiliation_sinistre)

print(
    "\nLes trois variables numériques les plus liées à la résiliation sont "
    "Score Risque (0-100), Nb Sinistres (3 ans) et Coeff. Bonus-Malus. "
    "Le taux de résiliation est le plus élevé après un vol (37 %), "
    "puis un bris de glace (27 %), contre 6 % lorsqu'il n'y a eu aucun sinistre."
)

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64

Les trois variables numériques les plus liées à la résiliation sont Score Risque (0-100), Nb Sinistres (3 ans) et Coeff. Bonus-Malus. Le taux de résiliation est le plus élevé après un vol (37 %), puis un bris de glace (27 %), contre 6 % lorsqu'il n'y a eu aucun sinistre.


In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.1 0.1


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer([
('num', StandardScaler(), num_cols),
('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
candidats = {
'Régression Logistique': LogisticRegression(
max_iter=1000, class_weight='balanced', random_state=42),
'Random Forest': RandomForestClassifier(
n_estimators=300, max_depth=4, min_samples_leaf=10,
class_weight='balanced', random_state=42),
}
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
for nom, algo in candidats.items()}

In [22]:
from sklearn.model_selection import cross_val_score

for nom, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train, y_train, cv=5, scoring='roc_auc'
    )
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

print(
    "\nL'écart entre les deux modèles n'est pas significatif : leurs performances se recouvrent "
    "compte tenu de leur écart-type. On retient le Random Forest, qui fournit en plus les "
    "importances de variables utiles pour l'interface."
)

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096

L'écart entre les deux modèles n'est pas significatif : leurs performances se recouvrent compte tenu de leur écart-type. On retient le Random Forest, qui fournit en plus les importances de variables utiles pour l'interface.


In [24]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
confusion_matrix, classification_report)
pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('F1 :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC :', round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.87
F1 : 0.519
ROC-AUC : 0.853
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100





R1. Un modèle qui prédit toujours reste  obtient 90 % d'accuraci, mais il ne détecte aucun client qui résilie. Il n'est donc pas meilleur que le modèle obtenu : l'accuraci seule est trompeuse en présence d'un déséquilibre de classes. Notre modèle accepte une légère baisse d'accuracy pour détecter une partie des résiliations.

R2. Pour le service Fidelisation, le faux négatif coute généralement le plus cher : c'est un client qui va résilier et que l'on n'a pas contacté. Un faux positif correspond à un appel inutile auprès d'un client qui serait resté ; il représente un coût commercial, mais souvent inférieur à la perte d'un client.

R. Il faut donc plutot baisser le seuil en dessous de 0,5. Le modèle classera davantage de clients comme susceptibles de résilier, ce qui augmente le rappel et réduit les faux négatifs, au prix d'une hausse des faux positifs. Le seuil optimal doit ensuite être choisi selon le coût réel d'un appel et la valeur d'un client retenu.

In [25]:
import joblib, os
joblib.dump(pipeline, 'models/pipeline_resiliation.pkl')
print(os.path.getsize('models/pipeline_resiliation.pkl') / 1024, 'Ko')

493.689453125 Ko


In [26]:
import json

meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {
        c: {
            'min': float(X[c].min()),
            'max': float(X[c].max()),
            'median': float(X[c].median())
        }
        for c in num_cols
    },
    'cat_values': {
        c: sorted(X[c].unique().tolist())
        for c in cat_cols
    },
}
with open('models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [27]:
import joblib

modele = joblib.load('models/pipeline_resiliation.pkl')

client = pd.DataFrame([{
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25,
    'Nb Sinistres (3 ans)': 3, 'Montant Sinistres (€)': 4200,
    'Score Risque (0-100)': 72, 'Type Contrat': 'Bronze',
    'Catégorie Prof.': 'Entrepreneur', 'Usage Véhicule': 'Professionnel',
    'Dernier Sinistre': 'Vol',
}])

print('Profil à risque')
print('Classe :', modele.predict(client))
print('Proba :', modele.predict_proba(client)[0, 1].round(3))

client_fidele = client.copy()
client_fidele['Ancienneté (mois)'] = 200
client_fidele['Nb Sinistres (3 ans)'] = 0
client_fidele['Type Contrat'] = 'Gold'
client_fidele['Dernier Sinistre'] = 'Aucun'

print('\nProfil fidèle')
print('Classe :', modele.predict(client_fidele))
print('Proba :', modele.predict_proba(client_fidele)[0, 1].round(3))

Profil à risque
Classe : [1]
Proba : 0.816

Profil fidèle
Classe : [1]
Proba : 0.561


In [28]:
try:
    modele.predict(client.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}


 Règle pour l'interface
Le pipeline exige exactement les 12 colonnes attendues, avec les mêmes noms et des types compatibles. Une colonne manquante, renommée ou supplémentaire peut empêcher la prédiction ; l'interface doit donc construire systématiquement le DataFrame selon le schéma décrit dans `metadata.json`.